In [1]:
from chapterloader import ChapterLoader

chapter_loader = ChapterLoader()
    
# Define paths and books
paths = {
    "docs/ganesh_puran": [
        "Ganesh_Puran_Krida_Khand",
        "Ganesh_Puran_Upasana_Khand"
    ],
    "docs/mudgal": [
        f"Mudgal_Puran_Khand_{i}" for i in range(1, 10)
    ]
}

# Load all documents
all_documents = chapter_loader.load_from_paths(paths)

print(f"\nTotal documents loaded: {len(all_documents)}")

# Example: Print first document
if all_documents:
    print("\nSample document:")
    print(f"Book: {all_documents[0].metadata['book']}")
    # print(f"File: {all_documents[0].metadata['file']}")
    print(f"Chapter: {all_documents[0].metadata['chapter_info']}")
    print(f"Content preview: {all_documents[0].page_content[:150]}...")


Loading from folder: docs/ganesh_puran
Loaded 155 chapters from Ganesh_Puran_Krida_Khand
Loaded 92 chapters from Ganesh_Puran_Upasana_Khand
Total chapters from this folder: 247

Loading from folder: docs/mudgal
Loaded 54 chapters from Mudgal_Puran_Khand_1
Loaded 74 chapters from Mudgal_Puran_Khand_2
Loaded 51 chapters from Mudgal_Puran_Khand_3
Loaded 52 chapters from Mudgal_Puran_Khand_4
Loaded 45 chapters from Mudgal_Puran_Khand_5
Loaded 45 chapters from Mudgal_Puran_Khand_6
Loaded 16 chapters from Mudgal_Puran_Khand_7
Loaded 50 chapters from Mudgal_Puran_Khand_8
Loaded 41 chapters from Mudgal_Puran_Khand_9
Total chapters from this folder: 428

Total documents loaded: 675

Sample document:
Book: Ganesh_Puran_Krida_Khand
Chapter: Chapter 1 : The Instruction of Narada
Content preview: Chapter 1 : The Instruction of Narada

The latter part of the Shri Ganesha Purana begins here.

Salutations to Shri Ganesha. Salutations to the great ...


In [2]:
len(all_documents)

675

In [3]:
all_documents[600]

Document(metadata={'book': 'Mudgal_Puran_Khand_8', 'chapter_info': 'Chapter 17 : The Granting of Boons to the Yugas'}, page_content='Chapter 17 : The Granting of Boons to the Yugas\n\nShaunaka said: O Suta, O noble one, you have narrated the sin-destroying, yoga-bestowing, and complete story. Having heard it, we, the twice-born, are satisfied. Having heard of the characteristics of the Yugas in the Puranas, we are astonished and have become filled with great doubts. In the Krita Yuga, women had independence, but in the Treta Yuga, Brahma imposed restrictions. In the Dvapara Yuga, Shvetaketu established the rule of one husband. How was the virtuous Ahalya cursed by Gautama in the Krita Yuga? We have seen many such paths in the Krita Yuga. It is said that in the Krita Yuga, there was no Varna-Ashrama system; people were described as belonging to a single Varna and Ashrama. In the Treta Yuga, the great-grandfather established the Kshatriya dharma, and from then on, people were described a

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(docs,chunk_size=1000,chunk_overlap=200):
    txt_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = txt_splitter.split_documents(docs)
    print(f"Split {len(docs)} documents into {len(split_docs)} chunks.")
    
    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:150]}...")
        print(f"Metadata: {split_docs[0].metadata}...")
    
    return split_docs


In [5]:
chunks = split_documents(all_documents)

Split 675 documents into 6096 chunks.

Example chunk: 
Content: Chapter 1 : The Instruction of Narada

The latter part of the Shri Ganesha Purana begins here.

Salutations to Shri Ganesha. Salutations to the great ...
Metadata: {'book': 'Ganesh_Puran_Krida_Khand', 'chapter_info': 'Chapter 1 : The Instruction of Narada'}...


In [6]:
chunks[1500]

Document(metadata={'book': 'Ganesh_Puran_Upasana_Khand', 'chapter_info': "Chapter 25 : The Death of King Chandrasena and Queen Sulabha's Grief"}, page_content='Consumed by grief and delusion, they bowed down, clutching his feet. Some took his hands and placed them upon their own heads. Others wept loudly, striking their faces and chests, while some collapsed as if dead, driven by intense affection. His wife, Sulabha, wept with a sorrowful voice, striking her heart with her hands in deep agony. With her ornaments scattered, she fainted and fell to the ground, where she was supported by the women of the city who shared her grief. The beautiful queen lamented, crying out, "O Lord, O Lord," without shame or restraint.')

In [9]:
from embedder import SentenceTransformerEmbeddings
from langchain_astradb import AstraDBVectorStore
import uuid
import os
import dotenv

dotenv.load_dotenv(override=True)

embedder = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2",
    device=None
)

vector_store = AstraDBVectorStore(
    collection_name="documents",
    embedding=embedder,
    token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
    api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT"),
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
import os
import dotenv

dotenv.load_dotenv(override=True)
from astrapy import DataAPIClient, Database


def connect_to_database() -> Database:
    endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
    token = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

    if not token or not endpoint:
        raise RuntimeError(
            "Environment variables API_ENDPOINT and APPLICATION_TOKEN must be defined"
        )

    client = DataAPIClient()

    database = client.get_database(endpoint, token=token)

    print(f"Connected to database {database.info().name}")

    return database

connect_to_database().get_collection(name="documents")

Connected to database puranas


Collection(name="documents", keyspace="default_keyspace", database.api_endpoint="https://5b97a1b0-ae80-4b96-acee-12fc891706fe-us-east-2.apps.astra.datastax.com", api_options=FullAPIOptions(token=StaticTokenProvider(AstraCS:QsqT...), ...))

In [11]:
if 'chunks' in locals():
    texts = [chunk.page_content for chunk in chunks]  # First 10 chunks
    metadatas = [chunk.metadata for chunk in chunks]
    ids = [str(uuid.uuid4()) for _ in range(len(texts))]
    
    doc_ids = vector_store.add_texts(texts, metadatas=metadatas, ids=ids)
    print(f"Added documents with IDs: {doc_ids[:3]}...")

Added documents with IDs: ['a188b080-7448-4b0b-85ac-9997ad005561', 'b9f183a6-0afb-4435-b16e-ca0f58337446', '22ce78d5-7595-4b18-81dc-d451656c02ee']...
